# Atelier Prompt Engineering — AI Business Assistant

**Contexte** : une entreprise souhaite mettre en place un « AI Business Assistant », un assistant IA polyvalent capable d'aider ses collaborateurs à exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés.

## Partie 1 — Anatomie d'un prompt

### Prompt

> Tu es un analyste satisfaction client senior pour une entreprise de services. Contexte : je te fournis un ensemble d'avis clients bruts collectés sur plusieurs canaux (email, réseaux sociaux, support). Tâche : analyse ces avis pour identifier le sentiment général, les thèmes récurrents (positifs et négatifs) et les points d'amélioration prioritaires. Contraintes : appuie chaque conclusion sur des éléments présents dans les avis, ne fais aucune supposition non justifiée, reste factuel et synthétique. Format de sortie : un résumé en 3 parties (Sentiment global, Thèmes récurrents, Recommandations), chacune sous forme de liste à puces.


### Composantes identifiées

| Composante | Contenu |
|---|---|
| **Rôle** | Analyste satisfaction client senior |
| **Contexte** | Avis clients bruts multi-canaux (email, réseaux sociaux, support) |
| **Tâche** | Identifier sentiment général, thèmes récurrents, points d'amélioration |
| **Contraintes** | S'appuyer uniquement sur les avis fournis, pas de suppositions, rester factuel |
| **Format de sortie** | 3 sections à puces : Sentiment global / Thèmes récurrents / Recommandations |


## Partie 2 — Comparer les techniques de prompting

Comparaison des techniques **zero-shot**, **one-shot**, **few-shot** et **prompt structuré** sur la classification du commentaire : *"Le service est rapide mais l'application plante régulièrement."*


### Prompt 1 — Zero-shot

> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 2 — One-shot

> Exemple : "Le personnel est agréable." -> positif
>
> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 3 — Few-shot

> Exemples :
> "Le personnel est agréable." -> positif
> "La livraison a eu 5 jours de retard." -> négatif
> "Le produit correspond à la description." -> neutre
>
> Classe le commentaire suivant dans une des catégories : positif, négatif, neutre. Réponds uniquement avec le mot de la catégorie, sans explication.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."

**Réponse obtenue :** négatif

### Prompt 4 — Structuré

> Rôle : classificateur de sentiment.
> Tâche : attribuer une seule classe parmi [positif, négatif, neutre] au commentaire suivant.
> Commentaire : "Le service est rapide mais l'application plante régulièrement."
> Contrainte : si le commentaire contient à la fois un aspect positif et un aspect négatif, privilégier la classe correspondant au problème le plus bloquant pour l'usage du service.
> Format de sortie : renvoyer uniquement le mot de la classe, sans justification.

**Réponse obtenue :** négatif

### Comparaison des résultats

| Technique | Réponse | Remarque |
|---|---|---|
| Zero-shot | négatif | Aucun exemple fourni ; le modèle s'appuie uniquement sur sa compréhension générale du sentiment |
| One-shot | négatif | Un seul exemple ; aide peu à trancher un cas mixte comme celui-ci |
| Few-shot | négatif | Exemples contrastés (positif/négatif/neutre) ; renforce la cohérence du critère de classification |
| Structuré | négatif | La contrainte explicite (« privilégier le problème le plus bloquant ») lève directement l'ambiguïté du commentaire mixte |

**Analyse** : une fois le format de sortie harmonisé entre les 4 prompts, les 4 techniques convergent vers la même classe (*négatif*) avec le même format concis (un seul mot). La différence entre les techniques ne se voit donc plus sur ce commentaire (non ambigu pour le modèle), mais elle deviendrait déterminante sur un cas plus ambigu : le zero-shot resterait le moins fiable (aucun repère), le few-shot améliorerait la cohérence du critère grâce aux exemples contrastés, et le prompt structuré resterait le plus robuste car il explicite directement la règle de décision plutôt que de la laisser déduire par le modèle à partir d'exemples.

---

## Partie 3 — Prompt Engineering et raisonnement

### 3.1 — Décomposition du prompt

**Prompt soumis au LLM :**

> Décompose le prompt suivant en sous-tâches, en expliquant à quoi sert chaque étape : "Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations."

**Réponse obtenue :**

On peut décomposer ce prompt en 4 sous-tâches principales :

| # | Sous-tâche | Description | But |
|---|---|---|---|
| 1 | Analyser les avis clients | Lire chaque avis et identifier ce que le client exprime : satisfaction, insatisfaction, plainte, suggestion, etc. | Comprendre le contenu brut des avis |
| 2 | Identifier les problèmes | Regrouper les avis qui parlent du même problème (ex. retards, bugs, mauvais accueil, prix élevés) | Faire ressortir les problèmes récurrents plutôt que traiter chaque avis séparément |
| 3 | Prioriser les problèmes | Déterminer quels problèmes sont les plus importants selon leur fréquence, leur gravité ou leur impact sur le client | Savoir sur quels problèmes agir en premier |
| 4 | Formuler des recommandations | Pour chaque problème prioritaire, proposer une action permettant de le résoudre ou de le réduire | Transformer l'analyse en plan d'action concret |

**Résumé du flux :** Avis bruts → Analyse → Problèmes → Priorisation → Recommandations

**Conclusion du LLM :** le prompt initial est assez vague — il demande directement le résultat final, alors qu'un prompt plus précis pourrait demander explicitement d'effectuer chacune de ces étapes.

---


### 3.2 — Analyse avec contraintes puis auto-vérification

**Avis client analysé :**

> "La livraison a pris 6 jours au lieu de 2 annoncés. Le support a mis 3 jours à répondre à mon email. Le produit en lui-même est correct."

#### Prompt A — Analyse avec contraintes

**Prompt soumis au LLM :**

> Voici un avis client : "La livraison a pris 6 jours au lieu de 2 annoncés. Le support a mis 3 jours à répondre à mon email. Le produit en lui-même est correct." Analyse cet avis et identifie le problème principal, puis propose une recommandation. Contrainte : ne mentionne que des faits présents dans l'avis.

**Réponse obtenue :**

> **Problème principal** : la livraison a pris 6 jours au lieu des 2 jours annoncés.
>
> **Recommandation** : réduire le délai de livraison afin qu'il corresponde au délai de 2 jours annoncé.

#### Prompt B — Auto-vérification (même conversation)

**Prompt soumis au LLM :**

> Vérifie la réponse que tu viens de donner à propos de cet avis client. Contrôle : les informations non justifiées par l'avis, les contradictions, les informations absentes de l'avis mais mentionnées dans ta réponse, les éventuelles hallucinations, et le respect de la contrainte ("ne mentionner que des faits présents dans l'avis").

**Réponse obtenue :**

| Point de contrôle | Résultat |
|---|---|
| Informations justifiées | « 6 jours » et « au lieu des 2 jours annoncés » sont bien présents dans l'avis |
| Contradictions | Aucune détectée |
| Hallucinations factuelles | Aucune — aucun nouveau chiffre, événement ou problème ajouté |
| Point problématique | « afin qu'il corresponde au délai de 2 jours annoncé » transforme le délai annoncé en objectif de recommandation — ce n'est pas un fait nouveau, mais une interprétation prescriptive |

**Conclusion du LLM :** la réponse est cohérente, mais pour respecter la contrainte de manière stricte, il vaut mieux formuler la recommandation uniquement à partir des faits explicitement constatés dans l'avis, sans y injecter une reformulation prescriptive du délai annoncé.

**Analyse (atelier)** : cet exercice illustre l'intérêt d'un second prompt de vérification : même une réponse globalement correcte peut contenir une légère dérive (ici, transformer un fait en objectif normatif) que l'auto-vérification permet de détecter et de documenter, sans nécessiter d'intervention humaine.

---


## Partie 4 — Sorties structurées

### 4.1 — Prompt pour sortie JSON structurée

**Prompt soumis au LLM :**

> Analyse le commentaire client suivant et retourne UNIQUEMENT un objet JSON valide, sans texte autour, avec exactement les champs suivants :
> - "sentiment" (string) : une valeur parmi "positif", "negatif", "neutre"
> - "categorie" (string) : le thème principal du commentaire (ex. "livraison", "produit", "support", "application")
> - "urgence" (string) : une valeur parmi "faible", "moyenne", "élevée"
> - "probleme" (string) : description courte du problème identifié, ou null si aucun
> - "confiance" (number) : score de confiance entre 0 et 1
>
> Commentaire : "Le délai de livraison annoncé n'a pas été respecté, j'attends toujours mon colis une semaine après."

**Réponse obtenue :**

```json
{"sentiment":"negatif","categorie":"livraison","urgence":"élevée","probleme":"Le délai de livraison annoncé n'a pas été respecté et le colis n'est toujours pas reçu une semaine après.","confiance":0.99}
```

| Champ | Valeur | Type |
|---|---|---|
| sentiment | negatif | string |
| categorie | livraison | string |
| urgence | élevée | string |
| probleme | Le délai de livraison annoncé n'a pas été respecté et le colis n'est toujours pas reçu une semaine après. | string |
| confiance | 0.99 | number |

**Analyse** : le LLM respecte bien la consigne « UNIQUEMENT un objet JSON valide, sans texte autour » — aucune phrase d'introduction ni de commentaire hors JSON. Il choisit *urgence: élevée* (plutôt que "moyenne"), ce qui est cohérent avec le ton du commentaire (« j'attends toujours ») marquant une insatisfaction plus forte qu'un simple retard signalé calmement.

---


### 4.2 — Règles de validation de la sortie JSON

**Prompt soumis au LLM :**

> Voici des règles de validation à respecter impérativement pour toute réponse JSON que tu génères :
> 1. Le JSON doit être syntaxiquement valide (aucun texte avant/après, guillemets doubles, pas de virgule finale).
> 2. Aucune propriété supplémentaire à celles demandées.
> 3. "sentiment" doit être exactement "positif", "negatif" ou "neutre".
> 4. "confiance" doit être un nombre compris entre 0 et 1 inclus.
> 5. "urgence" doit être exactement "faible", "moyenne" ou "élevée".
> Si une valeur ne peut pas être déterminée avec certitude, mets "confiance" à une valeur basse plutôt que de deviner.
>
> Applique ces règles pour vérifier ta réponse précédente à propos du commentaire sur le retard de livraison.

**Réponse obtenue :**

```json
{"sentiment":"negatif","categorie":"livraison","urgence":"moyenne","confiance":0.99}
```

### Validation par rapport aux règles

| Règle | Statut |
|---|---|
| JSON syntaxiquement valide | ✅ |
| Aucune propriété supplémentaire | ✅ |
| sentiment ∈ {positif, negatif, neutre} | ✅ (negatif) |
| confiance ∈ [0, 1] | ✅ (0.99) |
| urgence ∈ {faible, moyenne, élevée} | ✅ (moyenne) |

### Écart constaté par rapport à la réponse de la tâche 4.1

| Champ | Réponse 4.1 | Réponse 4.2 (après application des règles) |
|---|---|---|
| probleme | Présent (description du retard) | **Absent** de la réponse |
| urgence | élevée | **moyenne** |

**Analyse** : le LLM a fait disparaître le champ `probleme`, alors que rien dans les règles de validation ne demandait de le supprimer — les règles ne mentionnent que sentiment, confiance et urgence, sans rappeler explicitement la liste complète des champs attendus (sentiment, categorie, urgence, probleme, confiance). Cela montre une limite importante du prompt de validation : en ne redonnant pas la liste exhaustive des champs obligatoires, il laisse le LLM libre d'en omettre un, ce qui casserait le contrat de données côté application. Le changement de `urgence` (élevée → moyenne) illustre aussi que la confiance affichée (0.99) ne garantit pas la stabilité du jugement d'un appel à l'autre. Pour fiabiliser ce prompt, il faudrait lister explicitement les 5 champs obligatoires dans les règles de validation elles-mêmes, et pas seulement les contraintes de valeur.

---


## Partie 5 — Prompts pour les applications métier

### 5.1 — Résumé de document avec contraintes

**Document source utilisé** : rapport fictif *« L'intelligence artificielle en entreprise : bilan et perspectives »* (fichier `rapport_ia_entreprise.pdf`, joint au dépôt), portant sur l'adoption de l'IA par 180 entreprises de la région de Dakar entre 2023 et 2026.

**Prompt soumis au LLM :**

> Résume le document suivant en respectant ces contraintes : maximum 250 mots ; conserve toutes les informations factuelles importantes (chiffres, dates, noms) ; identifie clairement les objectifs du document ; identifie les résultats obtenus ; identifie les recommandations formulées ; n'invente aucune information absente du texte source. Structure la réponse en 4 sections : Résumé général, Objectifs, Résultats, Recommandations.
>
> Document : *(rapport complet — voir `rapport_ia_entreprise.pdf`)*

**Réponse obtenue :**

**Résumé général**
Le rapport analyse l'adoption de l'intelligence artificielle (IA) dans les entreprises sénégalaises et ouest-africaines entre 2023 et 2026, à partir d'une enquête menée auprès de 180 entreprises de la région de Dakar. Il montre une progression rapide de l'utilisation de l'IA, mais aussi des écarts importants entre secteurs.

**Objectifs**
- Mesurer le taux d'adoption de l'IA dans les secteurs bancaire, télécoms, agricole et distribution.
- Identifier les freins organisationnels et techniques.
- Évaluer l'impact perçu de l'IA sur la productivité et la qualité de service.

**Résultats**
En 2026, 42% des entreprises interrogées utilisent au moins un outil d'IA générative, contre 15% en 2023. L'adoption atteint 61% dans la banque, 54% dans les télécommunications, 37% dans la distribution et seulement 12% dans l'agriculture. Parmi les entreprises utilisatrices, 68% rapportent un gain de temps sur les tâches administratives et 34% une réduction des coûts du support client. Chez les non-utilisatrices, 29% citent le manque de compétences internes et 24% la confidentialité des données comme freins.

**Recommandations**
Le rapport recommande de renforcer la formation technique en IA dans les écoles et universités, d'encourager l'agriculture à expérimenter des usages prédictifs à faible coût et d'établir des lignes directrices sectorielles sur la protection des données, notamment dans les secteurs bancaire et télécoms.

**Analyse** : toutes les données chiffrées du document source (42%, 15%, 61%, 54%, 37%, 12%, 68%, 34%, 29%, 24%, 180 entreprises, période 2023-2026) sont fidèlement reprises, sans invention. La structure en 4 sections demandée est respectée, et la limite de 250 mots est tenue (~185 mots). Ce prompt illustre bien l'intérêt de contraintes explicites de fidélité factuelle pour un résumé destiné à un usage professionnel, où l'exactitude des chiffres prime sur la fluidité stylistique.

---


### 5.2 — Traduction FR → EN avec contraintes

**Document source** : le rapport complet `rapport_ia_entreprise.pdf`.

**Prompt soumis au LLM :**

> Traduis le document français suivant en anglais en respectant ces contraintes : conserve le sens exact du texte source ; conserve la structure (titres, paragraphes, listes) ; conserve les termes techniques sans les franciser/angliciser abusivement (garder les noms propres, acronymes et termes métier reconnus) ; ne résume pas le contenu, traduis-le intégralement ; ne rajoute aucune information, note ou commentaire qui n'est pas dans le texte source.
>
> Document : *(rapport complet — voir `rapport_ia_entreprise.pdf`)*

**Réponse obtenue :**

**Artificial Intelligence in Business: Review and Outlook**

This report provides an overview of the adoption of artificial intelligence (AI) in Senegalese and West African companies between 2023 and 2026, based on a survey conducted among 180 companies in the Dakar region. The objective is to assess the level of organizational maturity regarding AI, identify concrete uses already deployed, and formulate recommendations to accelerate responsible adoption.

**Objectives of the Study**
The study has three main objectives: first, to measure the AI adoption rate by business sector (banking, telecommunications, agriculture, retail); second, to identify the organizational and technical barriers encountered by companies; third, to assess the perceived impact of AI on productivity and service quality.

**Results Obtained**
Of the 180 companies surveyed, 42% report using at least one generative AI tool (conversational assistants, content generation) in their daily operations, compared with only 15% in 2023. The banking sector ranks first with an adoption rate of 61%, followed by telecommunications (54%) and retail (37%). The agricultural sector remains behind, with only 12% of the companies surveyed using AI tools, mainly for climate data analysis and yield forecasting. Among user companies, 68% report a perceived time saving on repetitive administrative tasks, and 34% state that they have reduced their customer support costs through the partial automation of first-level responses. Conversely, 29% of non-user companies cite a lack of internal skills as the main barrier, while 24% mention concerns related to data privacy.

**Recommendations**
Based on these results, three recommendations are formulated. First, strengthen technical AI training programs within local schools and universities in order to reduce the skills gap identified by nearly one-third of non-user companies. Next, encourage companies in the agricultural sector to experiment with low-entry-cost use cases, such as predictive analytics applied to data already collected by cooperatives. Finally, establish sector-specific guidelines on data protection when using generative AI tools, particularly in the banking and telecommunications sectors, where adoption is already the most advanced.

**Conclusion**
The adoption of AI in businesses is progressing rapidly in the Dakar region, driven by the most digitalized sectors, but remains uneven across business sectors. Bridging the skills gap and clarifying the data privacy framework appear to be the two priority levers for broader and more responsible adoption in the coming years.

**Analyse** : la structure du document source (titre, 4 sections, conclusion) est intégralement conservée dans la traduction. Tous les chiffres (42%, 15%, 61%, 54%, 37%, 12%, 68%, 34%, 29%, 24%, 180 entreprises) sont repris à l'identique. Aucun paragraphe n'est résumé ni raccourci, et aucune note de traducteur n'a été ajoutée — ce qui confirme le respect strict des contraintes du prompt, y compris sur un texte plus long testant réellement la conservation de structure.

---


### 5.3 — Classification d'un ticket informatique (JSON)

**Prompt soumis au LLM :**

> Classe le ticket informatique suivant dans une des catégories : réseau, logiciel, matériel, sécurité, accès, autre. Réponds uniquement en JSON valide avec les champs "categorie" (string, une valeur parmi la liste) et "justification" (string, une phrase courte expliquant le choix).
>
> Ticket : "Impossible de me connecter au VPN depuis ce matin, message d'erreur 'identifiants refusés'."

**Réponse obtenue :**

```json
{"categorie":"accès","justification":"Le ticket concerne un problème d'authentification avec des identifiants refusés lors de la connexion au VPN."}
```

| Champ | Valeur |
|---|---|
| categorie | accès |
| justification | Problème d'authentification (identifiants refusés) lors de la connexion au VPN |

**Analyse** : le cas est à la frontière entre « réseau » (le VPN étant un service réseau) et « accès » (le message d'erreur porte sur l'authentification). Le LLM tranche pour « accès » en se basant sur la nature de l'erreur (identifiants refusés) plutôt que sur le service concerné (VPN), ce qui est cohérent avec l'esprit de la classification. Pour fiabiliser ce choix sur des cas similaires, le prompt pourrait préciser un critère de départage explicite (ex. « si le problème concerne l'authentification, choisir 'accès' même si le service est un VPN/réseau »).

---


### 5.4 — Extraction d'informations depuis une facture (JSON)

**Prompt soumis au LLM :**

> Extrait les informations suivantes de la facture ci-dessous : numéro_facture, date, client, montant_ht, tva, montant_ttc. Retourne uniquement un JSON valide avec ces 6 champs. Si une information est absente du document, indique null pour ce champ (jamais de valeur inventée).
>
> Facture : "Facture n°FA-2026-0142 du 03/09/2026 — Client : SARL Diallo Services — Montant HT : 450 000 FCFA — TVA (18%) : 81 000 FCFA — Total TTC : 531 000 FCFA"

**Réponse obtenue :**

```json
{"numéro_facture":"FA-2026-0142","date":"03/09/2026","client":"SARL Diallo Services","montant_ht":"450 000 FCFA","tva":"81 000 FCFA","montant_ttc":"531 000 FCFA"}
```

| Champ | Valeur | Type retourné |
|---|---|---|
| numéro_facture | FA-2026-0142 | string |
| date | 03/09/2026 | string |
| client | SARL Diallo Services | string |
| montant_ht | 450 000 FCFA | string (avec devise) |
| tva | 81 000 FCFA | string (avec devise) |
| montant_ttc | 531 000 FCFA | string (avec devise) |

**Analyse** : toutes les valeurs sont correctement extraites, sans hallucination et sans champ manquant. Une limite est cependant à noter : les montants (`montant_ht`, `tva`, `montant_ttc`) sont retournés comme des **chaînes de caractères incluant la devise** ("450 000 FCFA"), alors qu'une application en aval attendrait probablement des **nombres purs** (450000) pour effectuer des calculs. Le prompt initial ne précisait pas explicitement le type attendu pour ces champs — contrairement à l'exercice de la Partie 4.1 où chaque champ était typé (string/number). Une version corrigée du prompt préciserait : *"montant_ht, tva, montant_ttc : nombres (number), sans texte ni symbole monétaire"*.

---


### 5.5 — Email d'excuses pour retard de livraison

**Prompt soumis au LLM :**

> Rédige un email destiné à un client dont la livraison a pris du retard. Objectifs : reconnaître le retard, présenter des excuses, expliquer la situation sans inventer de cause précise si elle n'est pas fournie, proposer une solution concrète. Ton : professionnel, courtois et rassurant. Longueur maximale : 150 mots.

**Réponse obtenue :**

> Bonjour,
>
> Nous vous informons que votre livraison a malheureusement pris du retard. Nous vous présentons nos sincères excuses pour ce contretemps et pour la gêne occasionnée.
>
> Nous comprenons l'importance de recevoir votre commande dans les délais prévus et regrettons cette situation. À ce stade, nous ne souhaitons pas vous communiquer une cause qui n'a pas encore été confirmée.
>
> Nous suivons attentivement l'acheminement de votre commande et vous tiendrons informé(e) de son évolution. Afin de vous apporter une solution concrète, nous vous proposons de vérifier ensemble le nouveau délai de livraison et de vous communiquer immédiatement toute mise à jour disponible.
>
> Merci pour votre compréhension et votre patience.
>
> Cordialement,
> Le service client

**Vérification des objectifs du prompt** :

| Objectif | Respecté ? |
|---|---|
| Reconnaître le retard | ✅ « votre livraison a malheureusement pris du retard » |
| Présenter des excuses | ✅ « nos sincères excuses pour ce contretemps » |
| Ne pas inventer de cause précise | ✅ « nous ne souhaitons pas vous communiquer une cause qui n'a pas encore été confirmée » — le LLM évite explicitement d'halluciner une cause |
| Proposer une solution concrète | ⚠️ Partiellement — la proposition (« vérifier ensemble le nouveau délai ») reste assez vague, sans action ferme (ex. remise, suivi prioritaire) |
| Ton professionnel, courtois, rassurant | ✅ |
| Longueur ≤ 150 mots | ✅ (~110 mots) |

**Analyse** : le point le plus intéressant est la façon dont le LLM gère la contrainte *« sans inventer de cause précise »* : plutôt que d'improviser une explication plausible, il choisit d'expliciter à l'utilisateur qu'aucune cause confirmée n'est communiquée — un comportement prudent qui respecte strictement la consigne anti-hallucination, au prix d'un email légèrement moins rassurant. La solution proposée reste toutefois assez générique ; un prompt plus directif pourrait imposer une liste de solutions concrètes possibles (ex. « propose un geste commercial parmi : remise, livraison prioritaire, remboursement partiel »).

---


## Partie 6 — Prompt Engineering pour le Machine Learning

*Dataset de référence : capteurs de consommation énergétique d'un bâtiment (température, humidité, consommation, occupation, horodatage).*

### 6.1 — Stratégies de nettoyage du dataset capteurs

**Prompt soumis au LLM :**

> Je travaille avec un dataset de capteurs pour prédire la consommation énergétique d'un bâtiment. Il contient les variables suivantes : température, humidité, consommation énergétique, occupation (occupé/inoccupé), horodatage.
>
> Propose une stratégie de traitement pour : les valeurs manquantes, les doublons, les valeurs aberrantes, les variables catégorielles. Pour chaque point, indique : la méthode de détection, le traitement recommandé, et les risques associés à ce traitement.

**Réponse obtenue :**

> *Pour un dataset de capteurs destiné à prédire la consommation énergétique, il faut être particulièrement prudent : certaines valeurs qui semblent « anormales » peuvent en réalité correspondre à un événement réel (forte chaleur, bâtiment vide, pic de consommation, panne de capteur, etc.).*

| Problème | Détection | Traitement recommandé | Risques associés |
|---|---|---|---|
| Valeurs manquantes | `isnull()`/`isna()`, taux de valeurs manquantes par variable, trous dans l'horodatage | Interpolation temporelle pour température/humidité si trous courts ; occupation imputée par la valeur précédente ; consommation interpolée seulement sur de très courts intervalles, sinon suppression ; suppression de la variable si trop de valeurs manquantes | L'interpolation peut lisser artificiellement et masquer de vrais changements ; supprimer des lignes réduit le dataset et peut créer des biais temporels |
| Doublons | `duplicated()` sur toutes les colonnes, et vérification des doublons par timestamp | Supprimer les doublons stricts ; pour plusieurs mesures au même timestamp avec des valeurs différentes, vérifier la fréquence réelle des capteurs avant d'agréger (mean, median...) | Une suppression aveugle peut éliminer de vraies mesures simultanées ; une mauvaise agrégation peut déformer la consommation |
| Valeurs aberrantes | IQR, Z-score, mais surtout règles métier (température physiquement impossible, humidité hors [0,100], consommation négative) et analyse graphique temporelle | Distinguer erreur de capteur et événement réel ; correction si règle fiable, sinon interpolation/médiane ou suppression pour une erreur évidente ; conserver un pic réel | L'IQR/Z-score peut classer un pic réel de consommation comme anomalie ; le supprimer ferait perdre les événements que le modèle doit justement apprendre à prédire |
| Variable catégorielle (occupation) | `value_counts()`, recherche de modalités incohérentes ("Occupé", "occupé", "OCCUPE"...) et valeurs manquantes | Nettoyer les modalités puis encoder (0 = inoccupé, 1 = occupé) ; vérifier la cohérence temporelle des changements d'état | Un encodage incohérent peut créer plusieurs catégories pour la même situation, ou introduire une fausse relation numérique |
| Horodatage | Vérification du format, timestamps invalides, doublons temporels, intervalles irréguliers, trous dans la série | Conversion en datetime, tri chronologique, extraction de features (heure, jour, mois...) ; séparation train/test chronologique | Une mauvaise gestion du temps peut provoquer une fuite de données (data leakage) si des informations futures sont utilisées pour imputer/normaliser les données d'entraînement |

**Pipeline recommandé par le LLM (ordre proposé)** :
1. Nettoyage temporel (datetime → tri chronologique → détection des trous)
2. Doublons (suppression stricte uniquement, examen avant toute agrégation)
3. Valeurs manquantes (mesure du taux par colonne, interpolation prudente, jamais de moyenne globale aveugle)
4. Valeurs aberrantes (règles physiques d'abord, IQR/Z-score en complément, vérification graphique, conservation des vrais pics)
5. Occupation (encodage binaire simple : inoccupé → 0, occupé → 1)
6. Variables temporelles dérivées du timestamp (heure, jour de semaine, weekend, mois), avec un encodage **cyclique** pour l'heure : `sin(2π × heure / 24)` et `cos(2π × heure / 24)`, pour refléter la proximité entre 23h et 0h qu'un encodage 0-23 ne capture pas

**Point clé soulevé par le LLM** : éviter le réflexe *« valeur aberrante → suppression »* — un bâtiment peut réellement passer de 2 kWh à 15 kWh (forte occupation, démarrage HVAC, pic de température). L'objectif n'est pas un dataset « parfaitement propre » mais un dataset fidèle au comportement réel du bâtiment. Le LLM recommande aussi un split train/test **chronologique** plutôt qu'aléatoire, pour éviter toute fuite d'information depuis le futur.

**Analyse (atelier)** : cette réponse va au-delà d'une simple liste de méthodes — elle apporte une réflexion critique propre au contexte métier (distinction erreur de capteur / événement réel) et une recommandation technique avancée (encodage cyclique de l'heure), ce qui montre l'intérêt de fournir un contexte précis (nature du dataset, objectif de prédiction) plutôt qu'une question générique de nettoyage de données.

---


### 6.2 — Visualisations pertinentes pour la consommation énergétique

**Prompt soumis au LLM :**

> Je travaille avec un dataset de capteurs pour prédire la consommation énergétique d'un bâtiment. Il contient les variables suivantes : température, humidité, consommation énergétique, occupation (occupé/inoccupé), horodatage.
>
> Propose les visualisations les plus pertinentes pour comprendre la consommation énergétique du bâtiment. Pour chaque visualisation, indique : le type de graphique, les variables utilisées, l'objectif, et l'interprétation attendue.

**Réponse obtenue :**

> *Pour ce dataset, l'objectif est de comprendre comment la consommation évolue dans le temps, comment elle dépend des conditions ambiantes et quel rôle joue l'occupation du bâtiment.*

| # | Visualisation | Variables | Objectif | Interprétation attendue |
|---|---|---|---|---|
| 1 | Courbe temporelle | Horodatage + consommation | Voir l'évolution dans le temps | Pics, creux, tendances, cycles journaliers/hebdomadaires |
| 2 | Consommation moyenne par heure | Heure + consommation | Étudier le profil quotidien | Hausse en heures d'activité, baisse la nuit |
| 3 | Consommation selon l'occupation | Occupation + consommation | Comparer bâtiment occupé vs inoccupé | Consommation plus élevée en période occupée → effet important de l'occupation |
| 4 | Boxplot consommation par occupation | Occupation + consommation | Comparer distribution, pas seulement moyennes | Médiane, dispersion, valeurs extrêmes pour les deux états |
| 5 | Température vs consommation | Température + consommation | Étudier la relation température/énergie | Relation possiblement non linéaire (chauffage/climatisation) |
| 6 | Humidité vs consommation | Humidité + consommation | Vérifier l'association humidité/consommation | Corrélation indiquant indirectement climatisation/ventilation |
| 7 | Matrice de corrélation | Température, humidité, consommation + variables dérivées | Vue globale des relations | Corrélation proche de ±1 = relation linéaire forte, mais pas de causalité prouvée |
| 8 | Courbe température + consommation | Horodatage + température + consommation | Comparer conditions et consommation simultanément | Vérifier si les pics coïncident avec certaines températures |
| 9 | Profil horaire occupé/inoccupé | Heure + occupation + consommation | Combiner temps et occupation | Habitudes de consommation selon l'état du bâtiment |
| 10 | Consommation par jour de semaine | Jour + consommation | Détecter écarts jours ouvrés/week-end | Consommation plus forte en semaine → impact de l'activité humaine |

**Les 5 visualisations prioritaires recommandées par le LLM (pour une EDA)** :
1. **Consommation dans le temps** — la plus importante selon le LLM : repère immédiatement pics, périodes creuses et anomalies
2. **Température vs consommation** — peut révéler une relation en U (consommation élevée aux températures extrêmes, chauffage/climatisation)
3. **Occupation vs consommation** — mesure l'importance de l'activité humaine en comparant les distributions
4. **Consommation moyenne par heure** — construit le profil énergétique journalier et identifie les heures de pointe
5. **Matrice de corrélation** — vérifie rapidement quelles variables numériques semblent les plus liées à la consommation avant modélisation

**Ordre d'EDA conseillé par le LLM** : horodatage → évolution temporelle → heure/jour → occupation → température → humidité → corrélations → anomalies → préparation du modèle, avec l'avertissement de ne pas se limiter à la corrélation : la temporalité et l'occupation sont probablement déterminantes pour ce problème.

**Analyse (atelier)** : la réponse dépasse la simple liste demandée en proposant une priorisation (5 visualisations clés) et un ordre logique d'exploration — ce qui montre que le LLM peut structurer une démarche d'analyse complète à partir d'une description de dataset, sans avoir accès aux données réelles.

---


### 6.3 — Modèles de prédiction de la consommation énergétique

**Prompt soumis au LLM :**

> Je travaille avec un dataset de capteurs pour prédire la consommation énergétique d'un bâtiment (variables : température, humidité, consommation, occupation, horodatage).
>
> Propose plusieurs modèles adaptés à la prédiction de la consommation énergétique. Pour chaque modèle, indique : le principe, les avantages, les limites, le type de problème, et les métriques pertinentes.

**Réponse obtenue :**

> *Pour ton problème, on est principalement face à un problème de régression supervisée, avec une dimension temporelle importante puisque la consommation dépend probablement de l'heure, du jour et des conditions précédentes.*

| Modèle | Principe | Avantages | Limites | Type de problème | Métriques |
|---|---|---|---|---|---|
| Régression linéaire | Relation linéaire entre variables et consommation | Simple, rapide, interprétable, bon modèle de référence | Ne capture pas les relations non linéaires | Régression | MAE, RMSE, R², MAPE |
| Ridge / Lasso | Régression linéaire régularisée | Réduit le surapprentissage ; Lasso peut sélectionner des variables | Reste essentiellement linéaire | Régression | MAE, RMSE, R² |
| Decision Tree Regressor | Divise les données selon des conditions (ex. température > 25°C) | Capture le non-linéaire, facile à interpréter | Surapprend facilement | Régression | MAE, RMSE, R² |
| Random Forest Regressor | Combine plusieurs arbres et moyenne les prédictions | Robuste, peu sensible aux outliers | Plus lourd, moins interprétable | Régression | MAE, RMSE, R², MAPE |
| Gradient Boosting | Arbres séquentiels corrigeant les erreurs précédentes | Très performant sur données tabulaires | Sensible aux hyperparamètres, plus lent | Régression | MAE, RMSE, R² |
| XGBoost / LightGBM / CatBoost | Gradient Boosting optimisé | Excellentes performances, gère les interactions complexes | Plus complexes à régler, moins interprétables | Régression | MAE, RMSE, R², MAPE |
| SVR | Fonction tolérant une marge d'erreur | Performant sur petits/moyens datasets | Peu adapté aux gros volumes, nécessite normalisation | Régression | MAE, RMSE, R² |
| KNN Regressor | Prédiction à partir des observations similaires | Simple, sans hypothèse forte | Lent avec beaucoup de données, sensible à l'échelle | Régression | MAE, RMSE, R² |
| MLP / réseau de neurones | Relations non linéaires via couches neuronales | Modélise des interactions complexes | Demande plus de données et de réglages | Régression | MAE, RMSE, R², MAPE |
| LSTM / GRU | Exploite l'historique temporel des consommations | Pertinent si l'historique est riche | Plus complexe, entraînement long, risque de surapprentissage | Série temporelle / régression | MAE, RMSE, MAPE, sMAPE |

**Benchmark progressif recommandé par le LLM** :
1. Baseline (consommation précédente / moyenne historique)
2. Régression linéaire (référence interprétable)
3. Random Forest (première approche non linéaire robuste)
4. XGBoost / LightGBM (candidats les plus prometteurs sur données tabulaires)
5. LSTM/GRU (si historique dense et objectif de prévision explicitement temporel)

**Point clé — variables temporelles et lags** : au-delà des features calendaires (heure, jour de semaine, mois, weekend, encodage cyclique sin/cos), le LLM recommande de créer des **variables de décalage (lags)** : `consommation_t-1` (heure précédente), `consommation_t-24` (même heure la veille), `consommation_t-168` (même heure une semaine avant) — ce qui peut nettement améliorer des modèles comme Random Forest ou XGBoost.

**Métriques recommandées, avec justification** :
- **MAE** : erreur absolue moyenne, facile à interpréter dans l'unité de consommation
- **RMSE** : pénalise davantage les grosses erreurs, utile si les fortes erreurs sont particulièrement problématiques
- **R²** : proportion de variance expliquée, utile pour comparer les modèles mais pas suffisant seul
- **MAPE** : erreur relative en %, pratique pour communiquer mais problématique si la consommation peut être nulle ou proche de zéro

**Recommandation finale du LLM** : XGBoost/LightGBM + variables temporelles + lags comme premier candidat sérieux ; LSTM/GRU pertinent seulement si l'objectif est explicitement une prévision séquentielle. Pour l'apprentissage, progression conseillée : Linear Regression → Decision Tree → Random Forest → Gradient Boosting/XGBoost, plutôt que de commencer directement par LSTM.

**Analyse (atelier)** : la réponse va au-delà d'un simple catalogue de modèles en proposant une méthodologie de benchmark progressif et une architecture expérimentale complète (feature engineering → modèles classiques/temporels → comparaison par métriques) — ce qui montre la valeur d'un prompt qui fournit un contexte métier précis plutôt qu'une question générique sur les algorithmes de régression.

---


### 6.4 — Explication des métriques de classification

**Prompt soumis au LLM :**

> Explique les métriques de classification suivantes : Accuracy, Precision, Recall, F1-score, ROC-AUC. Pour chacune, donne la définition, explique son interprétation, donne un exemple concret, et indique dans quel contexte elle est particulièrement utile.

**Réponse obtenue** (illustrée avec un modèle de détection de spam — TP=80, TN=890, FP=10, FN=20 sur 1000 emails) :

| Métrique | Définition | Valeur sur l'exemple | Interprétation | Contexte d'utilité |
|---|---|---|---|---|
| **Accuracy** | (TP+TN) / (TP+TN+FP+FN) | 97% | 97% des prédictions sont correctes | Utile sur classes équilibrées ; trompeuse sur classes déséquilibrées (un modèle qui prédit toujours "non-spam" sur 990 non-spams/10 spams atteindrait 99% sans détecter aucun spam) |
| **Precision** | TP / (TP+FP) | 88,9% | Quand le modèle prédit "spam", il a raison ~89% du temps | Cruciale quand les faux positifs sont coûteux (fraude, spam, diagnostic médical, modération de contenu) |
| **Recall** | TP / (TP+FN) | 80% | Sur 100 spams réels, le modèle en détecte 80 | Crucial quand les faux négatifs sont dangereux (détection de maladie, fraude, intrusions, alertes) |
| **F1-score** | 2 × (Precision×Recall)/(Precision+Recall) | ≈84,2% | Équilibre entre Precision et Recall ; pénalise un fort déséquilibre entre les deux | Utile sur classes déséquilibrées, quand Precision et Recall comptent toutes les deux |
| **ROC-AUC** | Aire sous la courbe TPR (Recall) vs FPR, à seuils variables | ex. 0,92 (bonne à excellente) | Probabilité qu'une observation positive reçoive un score plus élevé qu'une observation négative choisie au hasard | Utile pour comparer plusieurs modèles, évaluer la discrimination globale sans dépendre d'un seuil fixé |

**Exemple marquant fourni par le LLM sur le F1-score** (illustre pourquoi la moyenne harmonique pénalise le déséquilibre) :

| Precision | Recall | F1 approximatif |
|---|---|---|
| 90% | 90% | 90% |
| 99% | 50% | 66% |
| 50% | 99% | 66% |

**Grille de lecture ROC-AUC fournie** :

| ROC-AUC | Interprétation |
|---|---|
| 0.5 | Pas mieux que le hasard |
| 0.6–0.7 | Faible |
| 0.7–0.8 | Acceptable |
| 0.8–0.9 | Bonne |
| 0.9–1.0 | Excellente |
| 1.0 | Séparation parfaite |

**Règle mentale proposée par le LLM** : Precision → éviter les faux positifs · Recall → éviter les faux négatifs · F1 → équilibrer les deux · Accuracy → performance globale · ROC-AUC → capacité générale à séparer les classes.

**Recommandations métier données en exemple** : cancer → priorité au Recall (mieux vaut un faux positif qu'un malade manqué) ; fraude bancaire → Precision + Recall (F1 ou PR-AUC) ; spam → Precision importante pour ne pas supprimer de vrais emails ; classes équilibrées → Accuracy peut suffire ; comparaison de plusieurs modèles → ROC-AUC utile.

**Analyse (atelier)** : cette réponse est particulièrement solide car elle ne se contente pas de définitions abstraites — elle construit une matrice de confusion chiffrée cohérente sur tout l'exercice, illustre le piège classique de l'Accuracy sur classes déséquilibrées, et relie chaque métrique à des cas d'usage métier concrets (spam, fraude, médical), ce qui rend le point le plus important explicite : *le choix de la métrique dépend du coût relatif des erreurs (FP vs FN), pas d'une hiérarchie universelle entre métriques*.

---


### 6.5 — Explication des métriques de régression

**Prompt soumis au LLM :**

> Explique les métriques de régression suivantes : MAE, MSE, RMSE. Pour chacune, donne la définition, explique son interprétation, donne un exemple concret, et indique dans quel contexte elle est particulièrement utile.

**Réponse obtenue** (illustrée sur 3 observations de consommation énergétique) :

| Observation | Réelle | Prédite | Erreur |
|---|---|---|---|
| 1 | 100 kWh | 90 kWh | -10 |
| 2 | 200 kWh | 220 kWh | +20 |
| 3 | 300 kWh | 270 kWh | -30 |

| Métrique | Définition | Calcul sur l'exemple | Résultat | Unité |
|---|---|---|---|---|
| **MAE** | Moyenne des erreurs en valeur absolue | (10+20+30)/3 | **20 kWh** | Même unité que la cible |
| **MSE** | Moyenne des erreurs au carré | (100+400+900)/3 | **466,67 kWh²** | Unité au carré (moins intuitive) |
| **RMSE** | Racine carrée de la MSE | √466,67 | **≈21,60 kWh** | Même unité que la cible |

**Interprétations données par le LLM** :
- **MAE = 20 kWh** : en moyenne, le modèle se trompe de 20 kWh par prédiction — très intuitif car exprimé dans l'unité de la cible.
- **MSE** : élève les erreurs au carré, donc pénalise fortement les grosses erreurs (une erreur de 50 pèse 2500 contre 25 pour une erreur de 5) — mais son unité au carré la rend peu interprétable directement.
- **RMSE ≈ 21,6 kWh** : conserve la forte pénalisation des grosses erreurs de la MSE, tout en revenant dans l'unité d'origine, ce qui la rend plus lisible que la MSE tout en restant plus sensible aux outliers que la MAE.

**Tableau de synthèse MAE vs MSE vs RMSE** :

| Métrique | Principe | Unité | Grosses erreurs | Interprétation |
|---|---|---|---|---|
| MAE | Moyenne des erreurs absolues | Même unité | Peu pénalisées | Très facile |
| MSE | Moyenne des erreurs au carré | Unité² | Très pénalisées | Moins intuitive |
| RMSE | Racine de la MSE | Même unité | Très pénalisées | Facile à interpréter |

**Quand utiliser chaque métrique ?**
- **MAE** : métrique facile à interpréter, erreurs toutes considérées avec la même importance, utile en présence de valeurs aberrantes qu'on ne veut pas laisser dominer l'évaluation.
- **MSE** : quand les grosses erreurs sont particulièrement problématiques à pénaliser fortement, ou quand l'algorithme utilise nativement une fonction de coût quadratique.
- **RMSE** : quand les grosses erreurs comptent mais qu'on veut rester dans l'unité d'origine pour comparer plusieurs modèles — c'est la métrique la plus couramment utilisée en pratique.

**Exemple appliqué au projet énergétique (comparaison de modèles)** :

| Modèle | MAE | RMSE |
|---|---|---|
| Régression linéaire | 18 kWh | 25 kWh |
| Random Forest | 12 kWh | 19 kWh |
| XGBoost | 10 kWh | 15 kWh |

**Conclusion du LLM** : XGBoost a la meilleure MAE et la meilleure RMSE — la comparaison des deux permet en plus de juger l'importance relative des grosses erreurs pour chaque modèle. Règle pratique retenue : MAE = robuste et facile à comprendre ; MSE = pénalise fortement les grosses erreurs ; RMSE = comme MSE mais dans l'unité d'origine, donc plus interprétable.

**Analyse (atelier)** : cette réponse illustre bien les trois métriques sur un exemple chiffré cohérent et directement lié au cas d'usage du dataset capteurs (Partie 6), ce qui rend l'explication concrète plutôt qu'abstraite. Elle complète naturellement la Partie 6.4 (métriques de classification) : ensemble, ces deux tâches couvrent les métriques d'évaluation pour les deux grandes familles de problèmes ML (classification et régression) rencontrées dans l'AI Business Assistant.

---


## Partie 7 — Prompt Engineering et RAG

**Document de référence utilisé** : `politique_teletravail.pdf` (joint au dépôt) :

> « Politique de télétravail — Les collaborateurs peuvent télétravailler jusqu'à 2 jours par semaine, sous réserve de validation par leur manager. Les journées de télétravail doivent être déclarées via l'outil RH au moins 48h à l'avance. Le télétravail n'est pas ouvert aux collaborateurs en période d'essai. »

### Prompt A — question sans document

**Prompt soumis au LLM :**

> Combien de jours de télétravail par semaine sont autorisés dans cette entreprise ?

**Réponse obtenue :** 2 jours par semaine maximum.

### Prompt B — question avec document

**Prompt soumis au LLM :**

> Voici un document : « Politique de télétravail — Les collaborateurs peuvent télétravailler jusqu'à 2 jours par semaine, sous réserve de validation par leur manager. Les journées de télétravail doivent être déclarées via l'outil RH au moins 48h à l'avance. Le télétravail n'est pas ouvert aux collaborateurs en période d'essai. » Combien de jours de télétravail par semaine sont autorisés ?

**Réponse obtenue :** 2 jours de télétravail par semaine maximum.

### Prompt C — question avec document et contraintes anti-hallucination

**Prompt soumis au LLM :**

> Voici un document : « Politique de télétravail — Les collaborateurs peuvent télétravailler jusqu'à 2 jours par semaine, sous réserve de validation par leur manager. Les journées de télétravail doivent être déclarées via l'outil RH au moins 48h à l'avance. Le télétravail n'est pas ouvert aux collaborateurs en période d'essai. » Combien de jours de télétravail par semaine sont autorisés ? Contraintes : utilise uniquement le contexte fourni ; ne pas inventer une information absente ; signaler lorsqu'une information n'est pas trouvée et éventuellement citer la source ou le passage utilisé.

**Réponse obtenue :**

> Le document indique que les collaborateurs peuvent télétravailler jusqu'à 2 jours par semaine.
>
> Passage utilisé : « Les collaborateurs peuvent télétravailler jusqu'à 2 jours par semaine. »

### Comparaison des résultats

| Prompt | Document fourni | Contraintes anti-hallucination | Réponse | Traçabilité |
|---|---|---|---|---|
| A | Non | Non | 2 jours par semaine maximum | Aucune — impossible de savoir d'où vient ce chiffre |
| B | Oui | Non | 2 jours de télétravail par semaine maximum | Aucune citation ni référence au passage source |
| C | Oui | Oui | 2 jours par semaine | Passage exact cité entre guillemets |

**Analyse — un résultat surprenant et instructif** : contrairement à ce qu'on pourrait attendre, le **prompt A** (sans aucun document fourni) donne quand même la bonne réponse (« 2 jours par semaine »). Ce n'est pas parce que le modèle « connaît » la politique de l'entreprise — c'est une invention plausible, car « 2 jours de télétravail par semaine » est une politique très courante en entreprise, et le modèle génère une réponse statistiquement probable plutôt que d'admettre qu'il ne dispose d'aucune information réelle. **C'est exactement le risque d'hallucination que ce prompt A illustre** : une réponse peut être correcte par coïncidence, sans aucune garantie ni traçabilité, ce qui est dangereux en production — un chiffre différent (ex. 3 jours) aurait été tout aussi plausible et n'aurait pas pu être détecté comme faux sans le document réel.

Les prompts B et C donnent tous deux la bonne réponse une fois le document fourni, mais seul le **prompt C** offre une **traçabilité vérifiable** (citation exacte du passage), ce qui permet à un utilisateur ou à une application de contrôler la source de l'information plutôt que de faire confiance aveuglément à la réponse. C'est cette capacité à citer ses sources et à signaler l'absence d'information, plus que l'exactitude ponctuelle de la réponse, qui rend le prompt C fiable pour un usage en production dans un système RAG.

---


## Partie 8 — Évaluation et optimisation des prompts

**Document source utilisé** : le rapport complet `rapport_ia_entreprise.pdf` (déjà utilisé en Partie 5.1).

### Prompt A — instruction minimale

**Prompt soumis au LLM :**

> Résume ce texte.
>
> Document : *(rapport complet — voir `rapport_ia_entreprise.pdf`)*

**Réponse obtenue :**

**Résumé**
Le rapport analyse l'adoption de l'intelligence artificielle dans les entreprises sénégalaises et ouest-africaines entre 2023 et 2026, à partir d'une enquête menée auprès de 180 entreprises de la région de Dakar.

Les principaux résultats montrent une forte progression de l'utilisation de l'IA : 42% des entreprises utilisent désormais au moins un outil d'IA générative, contre seulement 15% en 2023. Le secteur bancaire est le plus avancé (61%), suivi des télécommunications (54%) et de la distribution (37%), tandis que l'agriculture reste en retard (12%).

L'IA apporte des bénéfices concrets : 68% des entreprises utilisatrices constatent un gain de temps sur les tâches répétitives et 34% ont réduit leurs coûts de support client grâce à l'automatisation. Cependant, le manque de compétences internes (29%) et les préoccupations concernant la confidentialité des données (24%) constituent des obstacles importants.

Le rapport recommande donc de :
1. Renforcer la formation en IA dans les écoles et universités.
2. Encourager l'expérimentation de l'IA dans l'agriculture, notamment avec l'analyse prédictive.
3. Mettre en place des règles sectorielles sur la protection des données, particulièrement dans la banque et les télécommunications.

En conclusion : l'adoption de l'IA progresse rapidement à Dakar, mais elle reste inégale selon les secteurs. Les deux priorités sont de réduire le déficit de compétences et de mieux encadrer la confidentialité des données.

**Analyse du prompt A** : sans aucune contrainte de longueur, le LLM produit un résumé complet et fidèle (~190 mots), mais logiquement plus long qu'un résumé contraint. Tous les chiffres du rapport sont conservés, et la structure suit fidèlement celle du document source (résumé, résultats, recommandations, conclusion) sans qu'on le lui ait demandé explicitement.

---


### Prompt B — avec contrainte de longueur

**Prompt soumis au LLM :**

> Résume ce texte en 150 mots.
>
> Document : *(rapport complet — voir `rapport_ia_entreprise.pdf`)*

**Réponse obtenue :**

Le rapport analyse l'adoption de l'intelligence artificielle (IA) dans les entreprises sénégalaises et ouest-africaines entre 2023 et 2026, à partir d'une enquête menée auprès de 180 entreprises de Dakar. L'étude montre une progression importante de l'IA générative : 42% des entreprises l'utilisent, contre 15% en 2023. Le secteur bancaire est le plus avancé avec 61% d'adoption, suivi des télécommunications (54%) et de la distribution (37%), tandis que l'agriculture reste à 12%. L'IA permet notamment de gagner du temps sur les tâches administratives et de réduire les coûts du support client. Toutefois, le manque de compétences internes et les préoccupations liées à la confidentialité des données constituent des freins majeurs. Le rapport recommande donc de renforcer la formation en IA, de développer des expérimentations dans l'agriculture et d'établir des lignes directrices pour protéger les données. L'adoption progresse, mais demeure inégale selon les secteurs.

*(≈135 mots — respecte la limite de 150 mots)*

**Analyse du prompt B** : la contrainte de longueur oblige le LLM à condenser le texte — les pourcentages de freins (29% et 24%) présents dans le prompt A disparaissent ici, remplacés par une formulation qualitative (« constituent des freins majeurs »). Les recommandations sont aussi fusionnées en une seule phrase au lieu d'une liste à 3 points. Le résumé reste fidèle et bien structuré, mais illustre un compromis : plus le résumé est contraint en longueur, plus certains détails chiffrés secondaires risquent d'être sacrifiés au profit des informations jugées prioritaires par le modèle.

---


### Prompt C — prompt enrichi (contraintes + structure)

**Prompt soumis au LLM :**

> Résume le texte suivant en respectant ces contraintes : maximum 150 mots ; conserve tous les chiffres et pourcentages ; distingue clairement le constat, les facteurs et les recommandations ; n'ajoute aucune information absente du texte source ; structure la réponse en 3 sections courtes (Constat, Facteurs, Recommandations).
>
> Document : *(rapport complet — voir `rapport_ia_entreprise.pdf`)*

**Réponse obtenue :**

**Constat**
Entre 2023 et 2026, une enquête auprès de 180 entreprises de Dakar montre que 42% utilisent au moins un outil d'IA générative, contre 15% en 2023. L'adoption atteint 61% dans la banque, 54% dans les télécommunications, 37% dans la distribution et 12% dans l'agriculture. Parmi les utilisatrices, 68% constatent un gain de temps et 34% une réduction des coûts du support client.

**Facteurs**
Le manque de compétences internes est cité par 29% des entreprises non utilisatrices. La confidentialité des données préoccupe 24% d'entre elles. L'adoption reste donc inégale selon les secteurs.

**Recommandations**
Le rapport recommande de renforcer les formations techniques en IA dans les écoles et universités, d'encourager l'agriculture à expérimenter des usages à faible coût, notamment l'analyse prédictive, et d'établir des lignes directrices sectorielles sur la protection des données, particulièrement dans la banque et les télécommunications.

*(≈110 mots — bien en dessous de la limite de 150 mots)*

**Analyse du prompt C** : contrairement au prompt B, **tous les chiffres sont conservés** (y compris les 29% et 24% de freins, perdus dans le résumé B), grâce à la contrainte explicite « conserve tous les chiffres et pourcentages ». La structuration en 3 sections nettes rend le résumé directement exploitable par une application (ex. affichage en blocs dans un tableau de bord).

---

### Évaluation comparative des trois versions

| Critère | Prompt A (minimal) | Prompt B (150 mots) | Prompt C (structuré) |
|---|---|---|---|
| Longueur | ~190 mots, non contrainte | ~135 mots | ~110 mots |
| Tous les chiffres conservés | ✅ (tous présents) | ❌ (29% et 24% perdus, remplacés par une formulation qualitative) | ✅ (tous présents, y compris 29% et 24%) |
| Structuration exploitable par une application | ⚠️ Paragraphe libre suivant la structure du document source | ⚠️ Paragraphe unique, peu structuré | ✅ 3 sections nettes (Constat/Facteurs/Recommandations) |
| Fidélité au texte source | ✅ | ✅ | ✅ |

**Conclusion de l'évaluation** : le prompt A produit le résumé le plus complet mais le plus long, sans structure imposée. Le prompt B, en ajoutant une simple contrainte de longueur, réduit le texte mais sacrifie des chiffres secondaires (29%, 24%) au profit de la concision. Le prompt C, avec des composants supplémentaires (conservation explicite des chiffres + structure de sortie imposée), est le seul à combiner à la fois **concision**, **fidélité factuelle complète** et **structure exploitable** — ce qui en fait la version la plus adaptée à une intégration dans l'AI Business Assistant.

---
